In [1]:
from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
processed_dir = Path("../data/processed")

matrice_tfidf = load_npz(processed_dir / "tfidf_matrix_unigrammes.npz")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
data = pd.read_csv(processed_dir / "sample_clean.csv")

# Il faut s'assurer que la matrice TF-IDF et les identifiants des lignes correspondent
# bien aux produits du DataFrame original.
assert matrice_tfidf.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(data["id"])

print("Dimensions :", matrice_tfidf.shape)

Dimensions : (500, 2039)


In [3]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=6,
    metric="cosine"
)

labels = dbscan.fit_predict(matrice_tfidf)

# Les résultats restent associés aux identifiants exportés.
clusters = ids.copy()
clusters["cluster"] = labels

display(clusters["cluster"].value_counts().sort_index())

nb_clusters = clusters.loc[
    clusters["cluster"] != -1, "cluster"
].nunique()

print("Nombre de clusters :", nb_clusters)

cluster
-1     335
 0      28
 1      55
 2       6
 3       6
 4       8
 5       6
 6       7
 7      15
 8       6
 9       7
 10     15
 11      6
Name: count, dtype: int64

Nombre de clusters : 12


In [4]:
sans_bruit = labels != -1

X_eval = matrice_tfidf[sans_bruit]
labels_evalues = labels[sans_bruit]

nb_clusters = len(np.unique(labels_evalues))

score = silhouette_score(
    X_eval,
    labels_evalues,
    metric="cosine"
)

print(f"Silhouette hors bruit : {score:.3f}")
print(f"Produits évalués : {sans_bruit.sum()} / {len(labels)}")
print("Bruit (%) :", (labels == -1).mean() * 100)

Silhouette hors bruit : 0.461
Produits évalués : 165 / 500
Bruit (%) : 67.0


In [5]:
projection_2d = TruncatedSVD(n_components=2, random_state=42)

In [6]:
projection_2d

,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary <random_state>`.",42
,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",2
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


In [7]:
coordonnees = projection_2d.fit_transform(matrice_tfidf)
visualisation = pd.DataFrame({
    "SVD 1": coordonnees[:, 0],
    "SVD 2": coordonnees[:, 1],
    "id": ids["id"].to_numpy(),
    "cluster": [
        "Bruit" if label == -1 else f"Cluster {label}"
        for label in labels
    ]
})

In [8]:
coeurs = np.zeros(len(labels), dtype=bool)
coeurs[dbscan.core_sample_indices_] = True

visualisation["statut"] = np.where(
    labels == -1,
    "Bruit",
    np.where(coeurs, "Cœur", "Bordure")
)

fig = px.scatter(
    visualisation,
    x="SVD 1",
    y="SVD 2",
    color="cluster",
    symbol="statut",
    symbol_map={"Cœur": "circle", "Bordure": "diamond", "Bruit": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — clusters et vrais points cœur",
    height=600,
    width=600
)

fig.show()

In [9]:
tsne = TSNE(
    n_components=2,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_tsne = tsne.fit_transform(matrice_tfidf.toarray())

visu_tsne = visualisation.copy()
visu_tsne["t-SNE 1"] = coord_tsne[:, 0]
visu_tsne["t-SNE 2"] = coord_tsne[:, 1]

fig = px.scatter(
    visu_tsne,
    x="t-SNE 1",
    y="t-SNE 2",
    color="cluster",
    symbol="statut",
    symbol_map={"Cœur": "circle", "Bordure": "diamond", "Bruit": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — visualisation t-SNE, distance cosinus",
    height=600,
    width=600
)

fig.show()

In [10]:
tsne_3d = TSNE(
    n_components=3,
    metric="cosine",
    perplexity=30,
    init="random",
    learning_rate="auto",
    random_state=42
)

coord_3d = tsne_3d.fit_transform(matrice_tfidf.toarray())

visu_3d = visualisation.copy()
visu_3d[["t-SNE 1", "t-SNE 2", "t-SNE 3"]] = coord_3d

fig = px.scatter_3d(
    visu_3d,
    x="t-SNE 1",
    y="t-SNE 2",
    z="t-SNE 3",
    color="cluster",
    symbol="statut",
    symbol_map={"Cœur": "circle", "Bordure": "diamond", "Bruit": "x"},
    color_discrete_map={"Bruit": "lightgray"},
    hover_data=["id", "statut"],
    opacity=0.7,
    title="DBSCAN — projection t-SNE en 3D",
    height=600,
    width=600
)

fig.update_traces(marker_size=4)
fig.update_layout(scene_aspectmode="data")
fig.show()

In [17]:
vectorizer = joblib.load("../models/tfidf_vectorizer_unigrammes.joblib")
mots = vectorizer.get_feature_names_out()

# Retrouver les descriptions dans l'ordre exact de la matrice
source = pd.read_csv(
    processed_dir / "sample_clean.csv"
).set_index("id")

lecture = ids.copy()
lecture["cluster"] = labels
lecture["products"] = source.loc[
    ids["id"], "products"
].to_numpy()
lecture["details"] = source.loc[
    ids["id"], "details"
].to_numpy()

assert matrice_tfidf.shape == (len(lecture), len(mots))

In [18]:
cluster_ids = np.unique(labels[labels != -1]) # Parcours tous les clusters sauf le bruit (-1)

for cluster_id in cluster_ids:
    masque = labels == cluster_id
    assert cluster_id != -1 and masque.any()

    X_cluster = matrice_tfidf[masque]

    # Poids TF-IDF moyen de chaque mot dans ce groupe
    poids = np.asarray(X_cluster.mean(axis=0)).ravel()
    top = np.argsort(poids)[::-1][:12]
    top = top[poids[top] > 0]

    mots_cluster = pd.DataFrame({
        "mot": mots[top],
        "poids TF-IDF moyen": poids[top]
    })

    fig = px.bar(
        mots_cluster,
        x="poids TF-IDF moyen",
        y="mot",
        orientation="h",
        title=f"Cluster {cluster_id} — mots dominants",
        height=400,
        width=600
    )
    fig.update_yaxes(categoryorder="total ascending")
    fig.show()

    # Descriptions les plus proches du centre moyen en distance cosinus
    centre = np.asarray(X_cluster.mean(axis=0))
    similarites = cosine_similarity(X_cluster, centre).ravel()

    positions = np.flatnonzero(masque)
    selection = positions[np.argsort(similarites)[::-1][:3]]

    print(f"Cluster {cluster_id} : {len(positions)} produits")

    with pd.option_context("display.max_colwidth", None):
        display(lecture.iloc[selection][["id", "details", "products"]])

Cluster 0 : 28 produits


,id,details,products
496,497,"capilene 2 fabric has an open knit that invites airflow and provides excellent wicking for endurance activities in warm to moderate temperatures fastest-drying of our capilene fabrics, it is insulates enough for use on chilly days and can be worn beneath other layers self-fabric neckline is soft next to skin raglan sleeves and underarm panels merge out of the way of pack straps flat-rib panel-piecing under the arms and sides is exceptionally breathable, fast drying and eliminates armpit seam chafing",Cap 2 crew
21,22,"capilene 2 fabric has an open knit that invites airflow and provides excellent wicking for endurance activities in warm to moderate temperatures fastest-drying of our capilene fabrics, it insulates enough for use on chilly days and can be worn beneath other layers self-fabric neckline soft against skin raglan sleeves and underarm panels merge out of the way of pack straps flat-rib panel-piecing under the arms and sides is exceptionally breathable, fast drying and eliminates armpit seam chafing",Cap 2 t-shirt
486,487,"silky soft capilene 1 is the lightest of our technical baselayer fabrics fast-drying, highly breathable and sleek, it provides 15-upf sun protection and can be worn either on its own or under layers in cooler temps rib-knit collar is soft against the skin raglan sleeves reduce chafe beneath pack straps cut long enough to tuck in",Cap 1 crew


Cluster 1 : 55 produits


,id,details,products
115,116,ringspun yarns for a softer hand taped shoulder seams screen-print inks are pvc- and phthalate-free,Rockpile t-shirt
321,322,ringspun yarns for a softer hand taped shoulder seams screen-print inks are pvc- and phthalate-free,Tarpon t-shirt
123,124,ringspun yarns for a softer hand taped shoulder seams screen-print inks are pvc- and phthalate-free,Squid t-shirt


Cluster 2 : 6 produits


,id,details,products
30,31,"large, zippered main compartment has fully padded, suspended laptop sleeve and separate internal mesh organizer sleeve second zippered compartment has office organizer pockets front catch-all sleeve with hidden zipper provides easy-access to documents, tickets and other important materials side organizer pocket with detachable key chain stretch-woven water bottle pocket fleece-lined top pocket for sunglasses or electronics 3-d spacer-mesh straps and back pad wick moisture, dry quickly and allow air circulation shoulder straps have slots for clip-on attachments adjustable sternum strap with built-in safety whistle web carrying handle stowable attachment straps holds 15 - 17 laptop",Crosstown
463,464,"flap-accessed main compartment, with hook-and-loop closure, has stretchwoven sleeve zippered organizer pocket with internal zippered security pocket zippered organizer pocket on the side of bag has detachable key chain easy-access document sleeve long, adjustable shoulder strap top-mounted carry handle fully padded to protect contents holds 17 laptop",Daily grind
353,354,"flap-accessed main compartment, with hook-and-loop closure, has large, zippered mesh pocket zippered organizer pocket with internal zippered security pocket zippered organizer pocket on the side of bag has a detachable key chain easy-access document sleeve long, adjustable shoulder strap top-mounted carry handle",Single shot


... (14 outputs left out)

Cluster 10 : 15 produits


,id,details,products
381,382,"2.5-layer tear-resistant fabric with a waterproof/breathable h2no barrier and a deluge dwr durable water repellent finish highly engineered interior surface texture channels moisture away from skin, protects the waterproof barrier from abrasion and slides easily over layers roll-down, 2-way-adjustable hood with a laminated visor for improved hood structure and visibility in poor conditions microfleece-lined neck and chin for comfort and enhanced protection of barrier all exterior zippers center front, handwarmer pockets and pit zips have a water-resistant coating to keep you dry without extra storm flaps and hook-and-loop closures waterproof handwarmer pockets - without mesh lining updated custom zipper pulls low-profile, laminated self-fabric cuff tabs drawcord hem",Rain shadow jkt
211,212,"2.5-layer tear-resistant fabric with a waterproof/breathable h2no barrier and a deluge dwr durable water repellent finish highly engineered interior surface texture channels moisture away from skin, protects the waterproof barrier from abrasion and slides easily over layers roll-down, 2-way-adjustable hood with a laminated visor for improved hood structure and visibility in poor conditions all zippers center-front, handwarmer pockets, and pit zips all feature a water resistant coating to keep you dry without extra storm flaps and hook-and-loop closures microfleece-lined neck and chin for comfort and enhanced protection of barrier waterproof handwarmer pockets- without mesh lining updated custom zipper pulls",Rain shadow jkt
326,327,2.5-layer fabric with waterproof/breathable h2no barrier and deluge dwr durable water repellent finish laminated visor 2-way-adjustable hood rolls down and stows microfleece-lined neck and chin for comfort and enhanced protection of barrier center zipper has exterior and interior storm flaps to keep water out pit zips with storm flaps and dwr-treated zippers for wet-weather protection self-fabric hook-and-loop cuff closures handwarmer pockets drawcord hem packs into zippered self-storage pocket,Torrentshell jkt


Cluster 11 : 6 produits


,id,details,products
483,484,"supple, soft blend of recycled nylon and spandex for total mobility, a light drape, fast dry time and abrasion resistance provides 40-upf sun protection partially elasticized waistband supports wide range of motion generous wide-entry drop-in front pockets left front pocket has an internal hidden zip security pocket additional welted front zip pocket on lower right pocket zippered welted back pockets gusseted crotch inseam is 32",Borderless trek pants
484,485,"supple, soft blend of recycled nylon and spandex for total mobility, a light drape, fast dry time and abrasion resistance provides 40-upf sun protection partially elasticized waistband supports wide range of motion pockets left front pocket has an internal hidden zip security pocket additional welted front zip pocket on lower right pocket zippered welted back pockets gusseted crotch zip-off legs convert pants to shorts pants inseam is 32 shorts inseam is 12 1/2",Borderless trek zip-off pants
14,15,"supple, soft blend of recycled nylon and spandex for total mobility, a light drape, fast dry time and abrasion resistance provides 40-upf sun protection partially elasticized waistband supports wide range of motion generous wide-entry front pockets left front pocket has a hidden internal zip security pocket welted zippered back pockets gusseted crotch for mobility inseam is 11",Borderless shorts-11 in.
